In [ ]:
"""
COMPLETE FIXED IMPLEMENTATION: ConvNeXtV2 + Focal Self-Attention for Skin Cancer Detection
"""

# ============================================
# CELL 1: Install and Import Libraries
# ============================================

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import glob
import warnings
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import time

warnings.filterwarnings('ignore')

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================
# CELL 2: Dataset Paths
# ============================================

DATA_ROOT = "/kaggle/input/datasets/msalmankhan231103/isic201970/isic2019_split"

TRAIN_PATH = os.path.join(DATA_ROOT, "train")
VAL_PATH = os.path.join(DATA_ROOT, "val")
TEST_PATH = os.path.join(DATA_ROOT, "test")

CLASS_NAMES = ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

print(f"Train path exists: {os.path.exists(TRAIN_PATH)}")
if os.path.exists(TRAIN_PATH):
    print(f"Found classes: {os.listdir(TRAIN_PATH)}")


# ============================================
# CELL 3: Model Components (FIXED)
# ============================================

class GlobalResponseNorm(nn.Module):
    """Global Response Normalization - Eq. (3)"""
    def __init__(self, dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, 1, dim))
        self.beta = nn.Parameter(torch.zeros(1, 1, 1, dim))

    def forward(self, x):
        # x: (B, H, W, C)
        norm = torch.norm(x, p=2, dim=(1, 2), keepdim=True)
        norm = norm / (norm.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * norm) + self.beta + x


class ConvNeXtV2Block(nn.Module):
    """ConvNeXtV2 Block - FIXED"""
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.grn = GlobalResponseNorm(4 * dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)
        
    def forward(self, x):
        residual = x
        # Depthwise conv
        x = self.dwconv(x)
        # Permute for LayerNorm
        x = x.permute(0, 2, 3, 1)  # (B, H, W, C)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        # Permute back
        x = x.permute(0, 3, 1, 2)  # (B, C, H, W)
        return x + residual


class FocalSelfAttention(nn.Module):
    """Simplified Focal Self-Attention - FIXED"""
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
    def forward(self, x):
        B, C, H, W = x.shape
        # Reshape for attention
        x = x.permute(0, 2, 3, 1)  # (B, H, W, C)
        
        qkv = self.qkv(x).reshape(B, H * W, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, H, W, C)
        x = self.proj(x)
        
        return x.permute(0, 3, 1, 2)  # (B, C, H, W)


class FocalSelfAttentionBlock(nn.Module):
    """Focal Attention Block"""
    def __init__(self, dim, num_heads=8, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = FocalSelfAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim)
        )
    
    def forward(self, x):
        # Attention block
        identity = x
        x = x.permute(0, 2, 3, 1)
        x = self.norm1(x)
        x = x.permute(0, 3, 1, 2)
        x = identity + self.attn(x)
        
        # MLP block
        identity = x
        x = x.permute(0, 2, 3, 1)
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2)
        return x + identity


class Downsample(nn.Module):
    """Fixed Downsampling Block"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)
    
    def forward(self, x):
        # x: (B, C, H, W)
        x = x.permute(0, 2, 3, 1)  # (B, H, W, C)
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2)  # (B, C, H, W)
        x = self.conv(x)
        return x


class ConvNeXtV2FocalAttention(nn.Module):
    """Complete Model - FIXED ARCHITECTURE"""
    def __init__(self, num_classes=8):
        super().__init__()
        
        # Dimensions from paper
        dims = [96, 192, 384, 768]
        depths = [4, 4, 12, 4]
        
        # Stem - FIXED: Proper shape handling
        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            nn.BatchNorm2d(dims[0]),  # Use BatchNorm instead of LayerNorm for spatial data
            nn.GELU()
        )
        
        # Stage 1: ConvNeXtV2 blocks
        self.stage1 = nn.Sequential(*[ConvNeXtV2Block(dims[0]) for _ in range(depths[0])])
        self.down1 = Downsample(dims[0], dims[1])
        
        # Stage 2: ConvNeXtV2 blocks
        self.stage2 = nn.Sequential(*[ConvNeXtV2Block(dims[1]) for _ in range(depths[1])])
        self.down2 = Downsample(dims[1], dims[2])
        
        # Stage 3: Focal Attention blocks
        self.stage3 = nn.Sequential(*[FocalSelfAttentionBlock(dims[2]) for _ in range(depths[2])])
        self.down3 = Downsample(dims[2], dims[3])
        
        # Stage 4: Focal Attention blocks
        self.stage4 = nn.Sequential(*[FocalSelfAttentionBlock(dims[3]) for _ in range(depths[3])])
        
        # Classification head
        self.norm = nn.LayerNorm(dims[3])
        self.head = nn.Linear(dims[3], num_classes)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    
    def forward(self, x):
        # Print shapes for debugging (remove after fixing)
        # print(f"Input: {x.shape}")
        
        x = self.stem(x)
        # print(f"After stem: {x.shape}")
        
        x = self.stage1(x)
        # print(f"After stage1: {x.shape}")
        x = self.down1(x)
        # print(f"After down1: {x.shape}")
        
        x = self.stage2(x)
        # print(f"After stage2: {x.shape}")
        x = self.down2(x)
        # print(f"After down2: {x.shape}")
        
        x = self.stage3(x)
        # print(f"After stage3: {x.shape}")
        x = self.down3(x)
        # print(f"After down3: {x.shape}")
        
        x = self.stage4(x)
        # print(f"After stage4: {x.shape}")
        
        # Global average pooling
        x = x.mean(dim=[2, 3])
        x = self.norm(x)
        x = self.head(x)
        
        return x
    
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================
# CELL 4: Dataset Loading
# ============================================

class SkinCancerDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), color='black')
        
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


def load_dataset(root_path):
    """Load dataset from folder structure"""
    paths, labels = [], []
    
    if not os.path.exists(root_path):
        print(f"Path does not exist: {root_path}")
        return paths, labels
    
    for class_name in os.listdir(root_path):
        class_dir = os.path.join(root_path, class_name)
        if os.path.isdir(class_dir):
            # Get class index
            if class_name not in CLASS_TO_IDX:
                print(f"Warning: Unknown class {class_name}")
                continue
            
            class_idx = CLASS_TO_IDX[class_name]
            img_files = glob.glob(os.path.join(class_dir, '*.*'))
            
            for img_file in img_files:
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    paths.append(img_file)
                    labels.append(class_idx)
            
            print(f"  {class_name}: {len(img_files)} images")
    
    return paths, labels


def get_transforms(img_size=224):
    """Data augmentation"""
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform


# ============================================
# CELL 5: Training Functions
# ============================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(loader), accuracy_score(all_labels, all_preds)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Validation'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return running_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels


# ============================================
# CELL 6: Main Training Pipeline
# ============================================

def run_experiment():
    """Main training pipeline"""
    
    # Configuration
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.01
    MOMENTUM = 0.9
    WEIGHT_DECAY = 2e-5
    
    print("="*60)
    print("LOADING DATASET")
    print("="*60)
    
    print("\nLoading training data:")
    train_paths, train_labels = load_dataset(TRAIN_PATH)
    print("\nLoading validation data:")
    val_paths, val_labels = load_dataset(VAL_PATH)
    print("\nLoading test data:")
    test_paths, test_labels = load_dataset(TEST_PATH)
    
    print(f"\nTotal - Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")
    
    if len(train_paths) == 0:
        print("ERROR: No training images found!")
        return None
    
    # Create data loaders
    train_transform, val_transform = get_transforms(224)
    
    train_dataset = SkinCancerDataset(train_paths, train_labels, transform=train_transform)
    val_dataset = SkinCancerDataset(val_paths, val_labels, transform=val_transform)
    test_dataset = SkinCancerDataset(test_paths, test_labels, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Initialize model
    print("\n" + "="*60)
    print("INITIALIZING MODEL")
    print("="*60)
    
    model = ConvNeXtV2FocalAttention(num_classes=8).to(device)
    num_params = model.get_num_params()
    print(f"Model parameters: {num_params:,} ({num_params/1e6:.2f}M)")
    
    # Optimizer and scheduler
    optimizer = optim.SGD(model.parameters(), 
                          lr=LEARNING_RATE, 
                          momentum=MOMENTUM, 
                          weight_decay=WEIGHT_DECAY,
                          nesterov=True)
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    
    # Training loop
    print("\n" + "="*60)
    print("STARTING TRAINING")
    print("="*60)
    
    best_val_acc = 0
    best_model_state = None
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Train
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        
        # Validate
        val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Update scheduler
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            print(f"*** New best model! Val Acc: {val_acc:.4f} ***")
    
    # Test evaluation
    print("\n" + "="*60)
    print("TEST EVALUATION")
    print("="*60)
    
    model.load_state_dict(best_model_state)
    test_loss, test_acc, test_preds, test_labels_list = validate(model, test_loader, criterion, device)
    
    # Metrics
    test_precision = precision_score(test_labels_list, test_preds, average='weighted', zero_division=0)
    test_recall = recall_score(test_labels_list, test_preds, average='weighted', zero_division=0)
    test_f1 = f1_score(test_labels_list, test_preds, average='weighted', zero_division=0)
    
    print(f"\nTest Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"Test Precision: {test_precision:.4f} ({test_precision*100:.2f}%)")
    print(f"Test Recall:    {test_recall:.4f} ({test_recall*100:.2f}%)")
    print(f"Test F1-Score:  {test_f1:.4f} ({test_f1*100:.2f}%)")
    
    # Compare with paper
    print("\n" + "="*60)
    print("COMPARISON WITH PAPER CLAIMS")
    print("="*60)
    paper_metrics = {'Accuracy': 93.60, 'Precision': 91.69, 'Recall': 90.05, 'F1': 90.73}
    our_metrics = {'Accuracy': test_acc*100, 'Precision': test_precision*100, 
                   'Recall': test_recall*100, 'F1': test_f1*100}
    
    for metric in paper_metrics:
        diff = our_metrics[metric] - paper_metrics[metric]
        print(f"{metric:10}: Paper={paper_metrics[metric]:.2f}%, Ours={our_metrics[metric]:.2f}%, Diff={diff:+.2f}%")
    
    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Curves')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
    axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
    axes[1].axhline(y=0.936, color='r', linestyle='--', label='Paper Target')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy Curves')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150)
    plt.show()
    
    # Confusion matrix
    cm = confusion_matrix(test_labels_list, test_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    
    print("\nClassification Report:")
    print(classification_report(test_labels_list, test_preds, target_names=CLASS_NAMES))
    
    return our_metrics


# ============================================
# CELL 7: RUN
# ============================================

if __name__ == "__main__":
    results = run_experiment()
    print("\n✅ Training completed!")

Using device: cuda
GPU: Tesla T4
Train path exists: True
Found classes: ['MEL', 'VASC', 'SCC', 'DF', 'NV', 'BKL', 'BCC', 'AK']
LOADING DATASET

Loading training data:
  MEL: 3165 images
  VASC: 177 images
  SCC: 439 images
  DF: 167 images
  NV: 9012 images
  BKL: 1836 images
  BCC: 2326 images
  AK: 606 images

Loading validation data:
  MEL: 904 images
  VASC: 50 images
  SCC: 125 images
  DF: 47 images
  NV: 2575 images
  BKL: 524 images
  BCC: 664 images
  AK: 173 images

Loading test data:
  MEL: 453 images
  VASC: 26 images
  SCC: 64 images
  DF: 25 images
  NV: 1288 images
  BKL: 264 images
  BCC: 333 images
  AK: 88 images

Total - Train: 17728, Val: 5062, Test: 2541

INITIALIZING MODEL
Model parameters: 52,758,056 (52.76M)

STARTING TRAINING

Epoch 1/50
LR: 0.010000


Validation: 100%|██████████| 159/159 [01:16<00:00,  2.08it/s]


Train Loss: 1.7241, Train Acc: 0.4482
Val Loss: 1.3355, Val Acc: 0.5251
*** New best model! Val Acc: 0.5251 ***

Epoch 2/50
LR: 0.009990


Validation: 100%|██████████| 159/159 [00:49<00:00,  3.23it/s]


Train Loss: 1.3053, Train Acc: 0.5218
Val Loss: 1.2745, Val Acc: 0.5348
*** New best model! Val Acc: 0.5348 ***

Epoch 3/50
LR: 0.009961


...

Validation: 100%|██████████| 159/159 [00:49<00:00,  3.22it/s]


Train Loss: 1.0503, Train Acc: 0.6207
Val Loss: 1.0213, Val Acc: 0.6270
*** New best model! Val Acc: 0.6270 ***

Epoch 11/50
LR: 0.009045


Validation: 100%|██████████| 159/159 [00:49<00:00,  3.23it/s]


Train Loss: 1.0399, Train Acc: 0.6230
Val Loss: 1.0318, Val Acc: 0.6254

Epoch 12/50
LR: 0.008853


Training:  69%|██████▉   | 384/554 [07:37<03:20,  1.18s/it, loss=0.9218]